# Error handling

> Controlling complexity is the essence of computer programming. --_Brian Kernighan_ [@kernighan1976softwaretools]

:::{admonition} Other resources
:class: resources

Dyalog docs
: [Error guards](https://dyalog.github.io/documentation/21.0/programming-reference-guide/defined-functions-and-operators/dfns-and-dops/error-guards/) · [Error codes](https://dyalog.github.io/documentation/21.0/programming-reference-guide/error-messages/apl-errors/) · [Signal event](https://dyalog.github.io/documentation/21.0/language-reference-guide/system-functions/signal/)

Webinar
: [Part 1](https://dyalog.tv/Webinar/?v=tDK0AKXXRAk) · [Part 2](https://dyalog.tv/Webinar/?v=PSrXyk5tN6o) · [Part 3](https://dyalog.tv/Webinar/?v=jkD-iQfjo_Q) · [Part 4](https://dyalog.tv/Webinar/?v=G7Ht8iKvDMo) · [Part 5](https://dyalog.tv/Webinar/?v=WP6R7VRmHfc)
:::

By now you will have encountered a range of errors where Dyalog takes objection to something you wrote, like for example


In [1]:
vec ← ⍳10
50⊃vec      ⍝ INDEX ERROR

INDEX ERROR
      50⊃vec      ⍝ INDEX ERROR
        ∧


If you're used to a Java or Python stack trace, the errors thrown by Dyalog will appear spartan, but after a bit of experience, they're actually _blissfully_ spartan, not unlike APL itself. As an aside, the K language is even more brutal in its simplicity here.

So how do you deal with errors? There is no `try-catch-throw` exception mechanism to draw on. What we have instead is the concept of an _error guard_:

In [2]:
Pick50 ← {
    3::'out of range' ⍝ Error guard, catching INDEX ERROR
    50⊃⍵
}

In [3]:
Pick50 ⍳10

out of range

An error guard takes an array of error numbers to the left, and an expression to be returned to the right, separated by two colons. Simple, right? Wrong.

Here's a rather contrived function that prepends a 1 to its left argument and then multiplies it with the inverse of its right argument. We want to protect against division by zero, in which case we just want to return the 1 prepended to the left argument, like so:

In [14]:
F ← {
    nums ← 1 ⍺
    0=⍵:nums
    nums×÷⍵
}

In [15]:
4 F 0
4 F 2

1 4

0.5 2

This seems like a job for an error guard. Division by zero in this case will throw a `DOMAIN ERROR`, which has a code of 11.

Try to predict what happens before revealing the actual result:

In [6]:
Fbug ← { ⍝ Can you spot the bug?
    11::nums
    nums←1 ⍺
    nums×÷⍵
}

In [7]:
4 Fbug 0

VALUE ERROR: Undefined name: nums
Fbug[1] 11::nums
            ∧


What happens is that the local environment is unwound _before_ the error-guard's body is evaluated. To get the behavior we wanted we'd need to do this:

In [8]:
Fnew ← { ⍝ Now correct
    nums←1 ⍺
    11::nums
    nums×÷⍵
}

In [9]:
4 Fnew 0

1 4

The other subtlety is that following the _setting_ of an error guard, subsequent calls disable tail-call optimization:

In [ ]:
G ← { 
    3::'out of range' ⍝ Error guard, catching INDEX ERROR
    h ⍺⊃⍵ ⍝ No longer a tail call, due to presence of error guard
}

This can be worked around by localizing the error guard in its own dfn:

In [11]:
G ← { 
    val ← ⍺ {3::'out of range' ⋄ ⍺⊃⍵} ⍵
    h val ⍝ Tail call
}

The mapping of numeric error code to error message can be found in Dyalog's documentation, referenced above, or via the system function `⎕EM`:

In [36]:
⎕EM⍳10 ⍝ ...and the rest

┌───────┬────────────┬───────────┬──────────┬────────────┬───────────┬────────────┬───────┬───────┬───────────┐
│WS FULL│SYNTAX ERROR│INDEX ERROR│RANK ERROR│LENGTH ERROR│VALUE ERROR│FORMAT ERROR│ERROR 8│ERROR 9│LIMIT ERROR│
└───────┴────────────┴───────────┴──────────┴────────────┴───────────┴────────────┴───────┴───────┴───────────┘

Error guards can also be stacked if you want to take different actions for different kinds of errors:

In [ ]:
G ← { 
    3::'handle index error'
    4::'handle rank error'
    11::'handle domain error'
    ⍝ error-prone function body here
}

or if you want a catch-all for the errors you think you might encounter, the guard takes a vector to the left:

In [ ]:
G ← { 
    3 4 11::'it all went pear-shaped in some way'
    ⍝ error-prone function body here
}

You can also (sort of) throw "exceptions". You may have noticed the definition for `Assert` we've used here and there already:

In [29]:
Assert ← {⍺←'assertion failure' ⋄ 0∊⍵:⍺ ⎕SIGNAL 8 ⋄ shy←0}

In [33]:
Assert 0=1

assertion failure
      assert 0=1
      ∧


The `⎕SIGNAL` ambivalent system function _throws_ an error of the number given by its right argument, presenting as the corresponding message from the vector given by `⎕EM` if no left argument is given, or by the left argument if given, as is the case in the `Assert` function above. In other words, we could for example claim that the "workspace is full" by throwing error 1, somewhat duplicitously:

In [32]:
⎕SIGNAL 1

WS FULL
      ⎕SIGNAL 1
      ∧


In the `Assert` function we make use of an error code that isn't used by Dyalog normally, error 8:

In [34]:
⎕SIGNAL 8

ERROR 8
      ⎕SIGNAL 8
      ∧
